In [ ]:
# ======================================================
# 🧠 CROSS-COHORT PROBABILITY SIMILARITY (ADNI vs CCNA)
# ======================================================

import pandas as pd
import numpy as np
from scipy.spatial.distance import jensenshannon
from scipy.stats import pearsonr
import json

# ------------------------------------------------------
# STEP 1: Load Probabilities
# ------------------------------------------------------
adni = pd.read_csv("adni_fusion_predictions.csv").filter(like="prob_class")
ccna = pd.read_csv("ccna_fusion_predictions.csv").filter(like="prob_class")

# ✅ Align sample count
n = min(len(adni), len(ccna))
adni = adni.iloc[:n]
ccna = ccna.iloc[:n]
print(f"✅ Loaded and aligned {n} samples across cohorts.")

# ------------------------------------------------------
# STEP 2: Align Class Dimensions
# ------------------------------------------------------
n_classes = min(adni.shape[1], ccna.shape[1])

if adni.shape[1] > ccna.shape[1]:
    print(f"⚠️ ADNI has {adni.shape[1]} classes, CCNA has {ccna.shape[1]} → trimming ADNI.")
    adni = adni.iloc[:, :n_classes]
elif ccna.shape[1] > adni.shape[1]:
    print(f"⚠️ CCNA has {ccna.shape[1]} classes, ADNI has {adni.shape[1]} → trimming CCNA.")
    ccna = ccna.iloc[:, :n_classes]

# ------------------------------------------------------
# STEP 3: Compute Mean Probability Distributions
# ------------------------------------------------------
adni_mean = adni.mean(axis=0).values
ccna_mean = ccna.mean(axis=0).values

# ✅ Clean and normalize
adni_mean = np.nan_to_num(adni_mean, nan=1e-9)
ccna_mean = np.nan_to_num(ccna_mean, nan=1e-9)
adni_mean /= adni_mean.sum() if adni_mean.sum() > 0 else 1
ccna_mean /= ccna_mean.sum() if ccna_mean.sum() > 0 else 1

# ------------------------------------------------------
# STEP 4: Compute Metrics
# ------------------------------------------------------
# 1️⃣ Jensen–Shannon
js_div = jensenshannon(adni_mean, ccna_mean)
js_sim = 1 - js_div

# 2️⃣ Bhattacharyya
bh_coeff = np.sum(np.sqrt(adni_mean * ccna_mean))

# 3️⃣ Hellinger
def hellinger_distance(p, q):
    return (1 / np.sqrt(2)) * np.sqrt(np.sum((np.sqrt(p) - np.sqrt(q)) ** 2))
hell_sim = 1 - hellinger_distance(adni_mean, ccna_mean)

# 4️⃣ Pearson
pear_corr, _ = pearsonr(adni_mean, ccna_mean)
pear_sim = (pear_corr + 1) / 2

# Composite
R_prob = np.mean([js_sim, bh_coeff, hell_sim, pear_sim])

# ------------------------------------------------------
# STEP 5: Report & Save
# ------------------------------------------------------
print("\n=== CROSS-COHORT SIMILARITY METRICS ===")
print(f"Jensen–Shannon Similarity:     {js_sim:.3f}")
print(f"Bhattacharyya Coefficient:     {bh_coeff:.3f}")
print(f"Hellinger Similarity:          {hell_sim:.3f}")
print(f"Pearson Similarity:            {pear_sim:.3f}")
print("-------------------------------------------")
print(f"Composite Relevance Index (Rₚᵣₒᵦ): {R_prob:.3f}")

results = {
    "Samples Used": int(n),
    "Classes Compared": n_classes,
    "Jensen–Shannon Similarity": float(js_sim),
    "Bhattacharyya Coefficient": float(bh_coeff),
    "Hellinger Similarity": float(hell_sim),
    "Pearson Similarity": float(pear_sim),
    "Composite R_prob": float(R_prob)
}
with open("cross_cohort_similarity_results.json", "w") as f:
    json.dump(results, f, indent=4)
print("\n💾 Saved results → cross_cohort_similarity_results.json")


✅ Loaded and aligned 31 samples across cohorts.
⚠️ ADNI has 4 classes, CCNA has 3 → trimming ADNI.

=== CROSS-COHORT SIMILARITY METRICS ===
Jensen–Shannon Similarity:     0.701
Bhattacharyya Coefficient:     0.900
Hellinger Similarity:          0.683
Pearson Similarity:            0.781
-------------------------------------------
Composite Relevance Index (Rₚᵣₒᵦ): 0.766

💾 Saved results → cross_cohort_similarity_results.json


In [ ]:
# ======================================================
# 🧠 CROSS-COHORT PROBABILITY SIMILARITY (ADNI vs CCNA)
# ======================================================

import pandas as pd
import numpy as np
from scipy.spatial.distance import jensenshannon
from scipy.stats import pearsonr, kendalltau
import json

# ------------------------------------------------------
# STEP 1: Load Probabilities
# ------------------------------------------------------
adni = pd.read_csv("adni_fusion_predictions.csv").filter(like="prob_class")
ccna = pd.read_csv("ccna_fusion_predictions.csv").filter(like="prob_class")

# ✅ Align sample count
n = min(len(adni), len(ccna))
adni = adni.iloc[:n]
ccna = ccna.iloc[:n]
print(f"✅ Loaded and aligned {n} samples across cohorts.")

# ------------------------------------------------------
# STEP 2: Align Class Dimensions
# ------------------------------------------------------
n_classes = min(adni.shape[1], ccna.shape[1])

if adni.shape[1] > ccna.shape[1]:
    print(f"⚠️ ADNI has {adni.shape[1]} classes, CCNA has {ccna.shape[1]} → trimming ADNI.")
    adni = adni.iloc[:, :n_classes]
elif ccna.shape[1] > adni.shape[1]:
    print(f"⚠️ CCNA has {ccna.shape[1]} classes, ADNI has {adni.shape[1]} → trimming CCNA.")
    ccna = ccna.iloc[:, :n_classes]

# ------------------------------------------------------
# STEP 3: Compute Mean Probability Distributions
# ------------------------------------------------------
adni_mean = adni.mean(axis=0).values
ccna_mean = ccna.mean(axis=0).values

# ✅ Clean and normalize
adni_mean = np.nan_to_num(adni_mean, nan=1e-9)
ccna_mean = np.nan_to_num(ccna_mean, nan=1e-9)
adni_mean /= adni_mean.sum() if adni_mean.sum() > 0 else 1
ccna_mean /= ccna_mean.sum() if ccna_mean.sum() > 0 else 1

# ------------------------------------------------------
# STEP 4: Compute Metrics
# ------------------------------------------------------
# 1️⃣ Jensen–Shannon
js_div = jensenshannon(adni_mean, ccna_mean)
js_sim = 1 - js_div

# 2️⃣ Bhattacharyya
bh_coeff = np.sum(np.sqrt(adni_mean * ccna_mean))

# 3️⃣ Hellinger
def hellinger_distance(p, q):
    return (1 / np.sqrt(2)) * np.sqrt(np.sum((np.sqrt(p) - np.sqrt(q)) ** 2))
hell_sim = 1 - hellinger_distance(adni_mean, ccna_mean)

# 4️⃣ Pearson
pear_corr, _ = pearsonr(adni_mean, ccna_mean)
pear_sim = (pear_corr + 1) / 2

# 5️⃣ Kendall’s τ
tau, tau_p = kendalltau(adni_mean, ccna_mean)
print(f"Kendall’s τ (rank correlation): {tau:.3f} (p = {tau_p:.4f})")

# Composite
R_prob = np.mean([js_sim, bh_coeff, hell_sim, pear_sim])
R_all = np.mean([js_sim, bh_coeff, hell_sim, pear_sim, max(tau, 0)])  # include only positive τ

# ------------------------------------------------------
# STEP 5: Report & Save
# ------------------------------------------------------
print("\n=== CROSS-COHORT SIMILARITY METRICS ===")
print(f"Jensen–Shannon Similarity:     {js_sim:.3f}")
print(f"Bhattacharyya Coefficient:     {bh_coeff:.3f}")
print(f"Hellinger Similarity:          {hell_sim:.3f}")
print(f"Pearson Similarity:            {pear_sim:.3f}")
print(f"Kendall’s τ (risk ranks):      {tau:.3f}")
print("-------------------------------------------")
print(f"Composite Rₚᵣₒᵦ: {R_prob:.3f}")
print(f"Composite Rₐₗₗ (+ τ): {R_all:.3f}")

results = {
    "Samples Used": int(n),
    "Classes Compared": n_classes,
    "Jensen–Shannon Similarity": float(js_sim),
    "Bhattacharyya Coefficient": float(bh_coeff),
    "Hellinger Similarity": float(hell_sim),
    "Pearson Similarity": float(pear_sim),
    "Kendall Tau": float(tau),
    "Composite R_prob": float(R_prob),
    "Composite R_all": float(R_all)
}
with open("cross_cohort_similarity_results.json", "w") as f:
    json.dump(results, f, indent=4)
print("\n💾 Saved results → cross_cohort_similarity_results.json")


✅ Loaded and aligned 31 samples across cohorts.
⚠️ ADNI has 4 classes, CCNA has 3 → trimming ADNI.
Kendall’s τ (rank correlation): 0.333 (p = 1.0000)

=== CROSS-COHORT SIMILARITY METRICS ===
Jensen–Shannon Similarity:     0.701
Bhattacharyya Coefficient:     0.900
Hellinger Similarity:          0.683
Pearson Similarity:            0.781
Kendall’s τ (risk ranks):      0.333
-------------------------------------------
Composite Rₚᵣₒᵦ: 0.766
Composite Rₐₗₗ (+ τ): 0.680

💾 Saved results → cross_cohort_similarity_results.json
